In [3]:
import pickle
import pandas as pd
import numpy as np
import os

In [62]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_df_oob_mid_pred'
disease = 'ICD10_E66'

In [63]:
with open(os.path.join(root,disease+'_pred.pkl'), 'rb') as f:
    data = pickle.load(f)

In [64]:
data

{'true_label': array([1, 1, 1, ..., 0, 0, 0]),
 'test_genes': Index(['P35580', 'O95239', 'P48058', 'Q13304', 'Q2VWP7', 'P84085', 'P05129',
        'Q9UQR1', 'P30041', 'O60229',
        ...
        'Q9H0X4', 'Q9H3K6', 'Q9HD64', 'Q9NS26', 'Q9NZ71', 'Q9ULR0', 'Q9ULZ0',
        'Q9Y2D5', 'Q9Y6F7', 'Q9Y6F8'],
       dtype='object', name='string_id', length=15432),
 'train_pos_genes': Index(['A8MTZ0', 'O00253', 'O00429', 'O00519', 'O00744', 'O00763', 'O00767',
        'O14520', 'O15118', 'O15156',
        ...
        'Q9P0J1', 'Q9UBU3', 'Q9UHB7', 'Q9UL25', 'Q9UPR6', 'Q9Y266', 'Q9Y2E4',
        'Q9Y6C9', 'Q9Y6K1', 'Q9Y6Q9'],
       dtype='object', name='string_id', length=254),
 'uniport_ppi_2019': array([-1.41560247, -1.65756717, -0.7115648 , ..., -1.2758199 ,
        -0.98593024, -1.0079364 ]),
 'ppi_2019_dw_40': array([-1.21696456, -1.43327435, -0.9145729 , ..., -1.14171964,
        -1.17876161, -1.27645461]),
 'diffusion_2019': array([-0.66487029, -0.67000302, -0.65251879, ..., -0.6688635

In [65]:
len(data['train_pos_genes'])

254

In [67]:
(-data['uniport_bio']).argsort().argsort()[:np.sum(data['true_label'])]

array([13959, 13963,  6111, 10046,  9445])

In [68]:
k = 10  # change to however many you want
indices = np.argpartition(data['uniport_seq'], -k)[-k:]

# Sort them by actual values (descending)
indices = indices[np.argsort(data['uniport_seq'][indices])[::-1]]
data['test_genes'][indices]

Index(['P01160', 'O15130', 'P06307', 'P07492', 'O00230', 'P01298', 'P23582',
       'P22466', 'P55089', 'P10092'],
      dtype='object', name='string_id')

In [30]:
train_num = []
test_num = []
disease = []
for file in os.listdir(root):
    disease.append(file[:9])

    with open(os.path.join(root,file), 'rb') as f:
        d_data = pickle.load(f)
    
    train_num.append(len(d_data['train_pos_genes']))
    test_num.append(np.sum(d_data['true_label']))

disease_dict = {
    'disease':disease,
    'train_num':train_num,
    'test_num':test_num
}
disease_sum = pd.DataFrame(disease_dict)


In [31]:
disease_sum

,disease,train_num,test_num
0,ICD10_C16,65,25
1,ICD10_F31,439,1
2,ICD10_C43,46,1
3,ICD10_I10,20,15
4,ICD10_J45,186,1
5,ICD10_I63,35,2
6,ICD10_C18,41,9
7,ICD10_G30,175,8
8,ICD10_E66,254,5
9,ICD10_M34,37,1


In [ ]:
abnorm_disease = ['ICD10_C23', 'ICD10_C43','ICD10_C50','ICD10_C53','ICD10_D83','ICD10_E66',
'ICD10_F01','ICD10_F31','ICD10_G20','ICD10_G24','ICD10_I25','ICD10_I50',
'ICD10_L40','ICD10_M34','ICD10_M41','ICD10_N04','ICD10_N17','ICD10_N46','ICD10_N80']

In [32]:
disease_sum[disease_sum['disease'].isin(abnorm_disease)]

,disease,train_num,test_num
1,ICD10_F31,439,1
2,ICD10_C43,46,1
8,ICD10_E66,254,5
9,ICD10_M34,37,1
13,ICD10_N17,83,5
14,ICD10_C50,483,61
16,ICD10_C53,12,3
17,ICD10_M41,25,2
18,ICD10_I25,113,25
20,ICD10_N04,70,3


In [47]:
import pandas as pd
import os
from features_reindex import get_feature, read_data, read_data_timecut
from model_diffusion import evaluate_disease
import pickle
import sys
import multiprocessing as mp
from sklearn.preprocessing import MinMaxScaler


root = '/itf-fi-ml/shared/users/ziyuzh/svm'

# time_spilt = True
# feature = 'ppi_'+str(time)

time_spilt = True
test_bug = True
# test_bug = False

if test_bug:
    # feature_list = ['uniport_ppi_2019','uniport_bio','uniport_seq','uniport_esm']
    # feature_list = ['ppi_2019','bioconcept']
    # feature_list = ['uniport_ppi_2017','ppi_2017_dw_80','uniport_exp','uniport_seq','uniport_esm']
    # feature_list = ['uniport_ppi_2017','ppi_2017_dw_80','uniport_exp','uniport_seq']
    # feature_list = ['uniport_ppi_2019','ppi_2019_dw_40','uniport_bio','uniport_seq','uniport_esm']
    feature_list = ['uniport_ppi_2019','ppi_2019_dw_40','uniport_bio','uniport_seq','uniport_esm','diffusion_2019']
    # feature_list = ['uniport_ppi_2019','ppi_2019_dw_40','diffusion_2019']
    # feature_list = ['uniport_ppi_2019','ppi_2019_dw_40','uniport_bio','uniport_seq','uniport_esm']


    # dga = 'opentarget'
    dga = 'disgenet'

    out_path = os.path.join(root,'results/temp')
    out_path_pred = out_path+'_pred/pred.pkl'
    time = 2019
else:
    feature_list = sys.argv[1].split(',')
    out_path = os.path.join(root,sys.argv[2])
    out_path_pred = out_path+'_pred'
    time = int(sys.argv[3])
    dga = sys.argv[4]

os.makedirs(out_path, exist_ok=True)
os.makedirs(out_path_pred, exist_ok=True)


merged_df = None

if time == 2017:
    time_feature_list = ['uniport_ppi_2017','ppi_2017_dw_80','uniport_exp','uniport_seq','uniport_esm']
elif time == 2019:
    time_feature_list = ['uniport_ppi_2019','ppi_2019_dw_40','uniport_bio','uniport_seq','uniport_esm','diffusion_2019']

for feature in time_feature_list:
    feature_df = get_feature(root, feature)

    if 'diffusion' in feature:
        pass
    else:
        feature_cols = [col for col in feature_df.columns if col.startswith('feature')]
        if feature_cols:
            scaler = MinMaxScaler()
            feature_df[feature_cols] = scaler.fit_transform(feature_df[feature_cols])

    # Rename columns starting with 'feature'
    feature_df.rename(columns={
        col: f"{feature}_{col}" if col.startswith('feature') else col
        for col in feature_df.columns
    }, inplace=True)

    # Merge iteratively to avoid keeping all DataFrames
    if merged_df is None:
        merged_df = feature_df
    else:
        merged_df = pd.merge(merged_df, feature_df, on='string_id', how='inner')
    del feature_df  # Free memory
name_list = feature_list + ['string_id']

merged_df = merged_df[[col for col in merged_df.columns if any(item in col for item in name_list)]]

if dga == 'disgenet':
    all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/dga_time_uniport.csv')
elif dga == 'opentarget':
    all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/ot_dga_time_uni.csv')
    all_df = all_df[all_df['score']>=0.4]

all_df = all_df[all_df['string_id'].isin(merged_df['string_id'])]
# all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/align_disgent_with_time.csv')

# methods = ['ooc','random_negative','pseudo_labeling','pseudo_labeling_mask']
# methods = ['random_negative','pseudo_labeling','pseudo_labeling_mask','pseudo_labeling_cluster_all_mask']
# methods = ['random_negative','random_negative_bagging','random_pos_negative_bagging']
methods = ['random_negative']

if time_spilt:
    selected_diseases = []
    for disease_id in all_df['disease_id'].unique():
        sub_df = all_df[all_df['disease_id']==disease_id]
        if len(sub_df) < 15:
            continue
        else:
            # print(type(time),type(sub_df['first_pub_year'].max()))
            if sub_df['first_pub_year'].max() > time and sub_df['first_pub_year'].min() <= time and len(sub_df[sub_df['first_pub_year']<time]) >=5:
                selected_diseases.append(disease_id)
else:
    selected_diseases = (
        all_df.groupby('disease_id')
        .filter(lambda x: (len(x) > 15))
        ['disease_id']
        .unique()
        .tolist())
print(feature_list, len(selected_diseases),len(merged_df))
all_results = []

['uniport_ppi_2019', 'ppi_2019_dw_40', 'uniport_bio', 'uniport_seq', 'uniport_esm', 'diffusion_2019'] 48 15686


In [49]:
disease = 'ICD10_E66'
if time_spilt:
    df, y = read_data_timecut(disease, all_df, merged_df,time)
else:
    df, y = read_data(disease, all_df, merged_df,time)

result_df = pd.DataFrame(columns=['method',"fold","para", 'top_recall_25','top_recall_300','top_recall_10%', 'top_precision_10%', 'max_precision_10%','top_recall_30%', 'top_precision_30%', 'max_precision_30%','pm_0.5%','pm_1%','pm_5%','pm_10%','pm_15%','pm_20%','pm_25%','pm_30%','auroc',"rank_ratio",'bedroc_1','bedroc_5','bedroc_10','bedroc_30'])

if time_spilt:
    test_idx = df[df['test']==1].index
    train_idx = df[y==1].index.difference(test_idx)
    df.drop(columns='test', inplace=True)

In [ ]:
import numpy as np
import pandas as pd
from sklearn import svm
from rdkit.ML.Scoring.Scoring import CalcBEDROC
# from pseudo_label import select_pseudo_negatives
from sklearn.metrics import roc_auc_score
import os
import pickle
import gseapy as gp
# from concurrent.futures import ProcessPoolExecutor
# import functools
from multiprocessing import Pool
from collections import defaultdict
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.neighbors import NearestNeighbors
from scipy.linalg import eigh
from sklearn.model_selection import StratifiedKFold
from scipy.stats import rankdata
from model_diffusion import compute_kernels, select_gamma_ratio, eval_bagging, neg_bagging


train_pos_df = df.loc[train_idx]
test_pos_df = df.loc[test_idx]
neg_num = 5*len(train_pos_df)
neg_df = df[y == 0]
neg_df_add_test_pos = pd.concat([neg_df, test_pos_df])


kernel_dir_path = os.path.join('/itf-fi-ml/shared/users/ziyuzh/svm/results/dw_auc_norm',str(time))

os.makedirs(kernel_dir_path, exist_ok=True)
kernel_pkl_path = os.path.join(kernel_dir_path,'path_save.pkl')

if os.path.isfile(kernel_pkl_path):
    print('kernels existing')
    with open(kernel_pkl_path, 'rb') as f:
        kernels_all_dict = pickle.load(f)
else:
    kernels_all_dict = dict()

add_feature_list = set(feature_list) - set(kernels_all_dict.keys())
if not add_feature_list:
    pass
else:
    add_feature_list = list(add_feature_list)
####### calculate full kernels for each feature and their logm
    print('calculating kernels...', add_feature_list)
    X_all = []
    
    for feature_name in add_feature_list:
        select_columns = [col for col in df.columns if col.startswith(feature_name)]
        X_all.append(df[select_columns].values)

    args_list = list(zip(X_all, add_feature_list, [kernel_dir_path] * len(X_all), [True] * len(X_all)))
    with Pool(min(len(add_feature_list), os.cpu_count(), 4)) as pool:
        # each tuple (X_feature, feature_id) is unpacked by starmap
        kernel_results = pool.starmap(
            compute_kernels,
            args_list)
    del X_all
    
    for fname, K_s_path_dict in kernel_results:
        kernels_all_dict[fname] = K_s_path_dict
        
    with open(kernel_pkl_path, 'wb') as f:
        pickle.dump(kernels_all_dict, f)
############################## cv get best gamma
args_list = [(neg_df, neg_num, train_pos_df, df, kernels_all_dict[fname], fname)
    for fname in feature_list]

with Pool(processes=len(feature_list)) as pool:
    best_ratios = pool.map(select_gamma_ratio, args_list)

best_ratios_dict = dict()
agg_feature = []
for fname, best_params, best_bedroc, best_auc in best_ratios:
    print(fname, best_params, best_bedroc, best_auc)
    best_ratios_dict[fname] = best_params
    # if best_auc > 0.67 and best_bedroc > 0.5:
    agg_feature.append(fname)
print('collect valid feature: ', agg_feature)
######################### using precalculated kernels to train svm and evaluate, get weights for kernels


kernels existing
uniport_ppi_2019 {'C_num': 1, 'gamma_ratio': 4, 'gamma': '0.2526347763086942'} 0.7757448355256811 0.8718094586512066
ppi_2019_dw_40 {'C_num': 1, 'gamma_ratio': 8, 'gamma': '0.12034508683354062'} 0.7748225406898882 0.87949150633535
uniport_bio {'C_num': 1, 'gamma_ratio': 2, 'gamma': '0.5427586367192363'} 0.7445239868841823 0.8232063408350986
uniport_seq {'C_num': 3, 'gamma_ratio': 8, 'gamma': '0.05611004556505588'} 0.46963724614337804 0.6918666677161966
uniport_esm {'C_num': 3, 'gamma_ratio': 2, 'gamma': '0.11355235622105712'} 0.4978056059510492 0.723666409781719
diffusion_2019 {'C_num': 3, 'gamma_ratio': '2', 'gamma': '2'} 0.7880064773246437 0.883479784928825
collect valid feature:  ['uniport_ppi_2019', 'ppi_2019_dw_40', 'uniport_bio', 'uniport_seq', 'uniport_esm', 'diffusion_2019']


In [52]:
# print('evaluation')

test_neg_df = neg_df
test_df = pd.concat([test_pos_df, test_neg_df])
test_index_loc = df.index.get_indexer(test_df.index)
y_test = np.array([1] * len(test_pos_df) + [0] * len(test_neg_df))


# # test_indices = test_df.index.values
# # enrich_train_genes = train_pos_df.index.values
# # enrich_train_set = enriched_set(enrich_train_genes,time)

num_processes = 20
base_seed = 42
seed_list = [base_seed + i for i in range(num_processes)]

# pathway_overlap_dict = dict()

rank_results_per_feature = dict()
predcition_collection = dict()
predcition_collection['true_label'] = y_test
predcition_collection["test_genes"] = test_df.index
predcition_collection["train_pos_genes"] = train_pos_df.index

In [ ]:
feature_name = 'uniport_bio'
gamma = best_ratios_dict[feature_name]['gamma_ratio']
X_path = kernels_all_dict[feature_name][gamma][0]
C_num = best_ratios_dict[feature_name]['C_num']

args_list = [
    (neg_df_add_test_pos, neg_num, train_pos_df, df, X_path, C_num, test_index_loc, seed)
    for seed in seed_list]


In [60]:
test_df

,uniport_ppi_2019_feature_1,uniport_ppi_2019_feature_2,uniport_ppi_2019_feature_3,uniport_ppi_2019_feature_4,uniport_ppi_2019_feature_5,uniport_ppi_2019_feature_6,uniport_ppi_2019_feature_7,uniport_ppi_2019_feature_8,uniport_ppi_2019_feature_9,uniport_ppi_2019_feature_10,...,uniport_esm_feature_1271,uniport_esm_feature_1272,uniport_esm_feature_1273,uniport_esm_feature_1274,uniport_esm_feature_1275,uniport_esm_feature_1276,uniport_esm_feature_1277,uniport_esm_feature_1278,uniport_esm_feature_1279,diffusion_2019_feature_0
string_id,,,,,,,,,,,,,,,,,,,,,
P35580,0.350539,0.486089,0.456312,0.484576,0.581848,0.528086,0.614201,0.670894,0.324722,0.473246,...,0.410706,0.470724,0.339166,0.597625,0.390310,0.037203,0.552610,0.559948,0.669999,3635
O95239,0.415784,0.372735,0.560816,0.535166,0.530207,0.762500,0.542288,0.472015,0.278881,0.561737,...,0.583971,0.616404,0.245213,0.453410,0.518728,0.464152,0.718466,0.478440,0.401053,11842
P48058,0.487222,0.316097,0.311748,0.525243,0.641814,0.475138,0.527431,0.264298,0.476001,0.362194,...,0.388961,0.692248,0.340966,0.324792,0.541938,0.786584,0.521037,0.394400,0.471583,4267
Q13304,0.439004,0.208164,0.553741,0.752631,0.675626,0.575604,0.516706,0.582376,0.505131,0.455864,...,0.631745,0.386135,0.485495,0.178249,0.783250,0.540754,0.689735,0.350225,0.373794,3806
Q2VWP7,0.699876,0.612886,0.447285,0.363098,0.605519,0.743365,0.688901,0.477487,0.382347,0.291346,...,0.414575,0.624030,0.332480,0.320598,0.544379,0.747637,0.606169,0.418932,0.428101,13058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Q9ULR0,0.572875,0.391738,0.429927,0.478681,0.548084,0.482649,0.651260,0.646052,0.495618,0.420106,...,0.315904,0.705466,0.335872,0.537480,0.455120,0.538185,0.691310,0.545430,0.540128,60
Q9ULZ0,0.500518,0.517299,0.569584,0.526759,0.491658,0.553639,0.556628,0.574510,0.536307,0.683537,...,0.804944,0.531245,0.667289,0.222340,0.762492,0.380301,0.835892,0.286713,0.401435,61
Q9Y2D5,0.571826,0.491911,0.527785,0.406249,0.381433,0.541702,0.461722,0.449068,0.577756,0.435595,...,0.700981,0.445173,0.265371,0.458775,0.489968,0.311359,0.676245,0.598528,0.405981,62


In [54]:
with Pool(processes=num_processes) as pool:
    bagging_y_scores_with_mask = pool.map(neg_bagging, args_list)

arrays = np.stack([arr for arr, _ in bagging_y_scores_with_mask])          # shape: (n, d)

# Build mask matrix (True = masked / skip)
mask = np.zeros_like(arrays, dtype=bool)
for i, (_, m) in enumerate(bagging_y_scores_with_mask):
    mask[i, m] = True

# Invert mask: True where we keep
keep = ~mask

# Compute sum and count efficiently
sum_arr = np.where(keep, arrays, 0).sum(axis=0)
count_arr = keep.sum(axis=0)

final_y_score = np.array(sum_arr / count_arr)

In [ ]:
final_y_score

array([-1.44943085, -1.44996961, -1.01388661, ..., -1.45115675,
       -1.15990672, -1.25118487])

In [58]:
for single_bag in bagging_y_scores_with_mask:
    print((-single_bag[0]).argsort().argsort())

[13971 14075  4869 ... 13374  8606  8804]
[14833 13160  3157 ... 13049 11544 12568]
[14019 11584  6320 ... 14690  8302  7420]
[14465 14019  7315 ... 15100  9309 12434]
[13597 14068  9275 ... 14181  7642 10130]
[12714 14737  4013 ... 13523 11277 11879]
[15208 12301  7202 ... 14685  8880 11066]
[11637 13485  5559 ... 13191  8984  9338]
[13598 14532  7516 ... 14499  9929  9499]
[13046 11692  3781 ... 13374  8677 13592]
[13930 14366  4713 ... 14174  9357  9561]
[12733 12555  3688 ... 14602 10702 12163]
[14298 14904  6693 ... 13499  8135  9980]
[13338 13721  7739 ... 12931  9055 10047]
[13132 13318  7220 ... 14016  4375  7928]
[13678 12760  7158 ... 12601  8619 10846]
[14222 14154  5267 ... 11743  8577 10132]
[11953 13810  7953 ... 13322  5758  8681]
[14849 14449  8854 ... 11186  8839 12179]
[13659 12692  9886 ... 12642 10642 10444]
